# Day 2 — Cross-Validation Practice
## Dataset: Loan Approval Prediction

Is notebook mein hum **Cross-Validation (CV)** practice karenge, `loan_approval_data.csv` par (Day-1 wali feature engineering yahan bhi thori repeat ki gayi hai taake yeh notebook standalone chal sake).

**Notebook Structure:**
1. Import Libraries
2. Load Dataset
3. Quick Preprocessing + Feature Engineering (recap)
4. Baseline: Simple Train/Test Split
5. K-Fold Cross-Validation
6. Stratified K-Fold Cross-Validation
7. Cross-Validation with Multiple Models (comparison)
8. Leave-One-Out vs K-Fold (concept + small demo)
9. `cross_val_score` vs `cross_validate` (multiple metrics)
10. Conclusion


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, LeaveOneOut,
    cross_val_score, cross_validate
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline


## 2. Load Dataset

In [ ]:
df = pd.read_csv('loan_approval_data.csv')
df = df.drop(columns=['Loan_ID'])
df.head()


## 3. Quick Preprocessing + Feature Engineering (Recap)

Yeh wahi steps hain jo Day-1 (Feature Engineering) notebook mein detail se cover kiye the — yahan sirf compact form mein taake dataset model-ready ho jaye.


In [ ]:
# Missing values
numeric_missing = ['LoanAmount', 'Loan_Amount_Term', 'Credit_History']
categorical_missing = ['Gender', 'Married', 'Dependents', 'Self_Employed']

for col in numeric_missing:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_missing:
    df[col] = df[col].fillna(df[col].mode()[0])

# Feature engineering
df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['Loan_Income_Ratio'] = df['LoanAmount'] / (df['Total_Income'] + 1)
df['EMI'] = df['LoanAmount'] / (df['Loan_Amount_Term'] + 1)
df['Balance_Income'] = df['Total_Income'] - (df['EMI'] * 1000)
df['Dependents_Num'] = df['Dependents'].replace({'3+': '3'}).astype(int)
df = df.drop(columns=['Dependents'])

# Encoding
binary_cols = ['Gender', 'Married', 'Education', 'Self_Employed']
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col].astype(str))

df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})
df = pd.get_dummies(df, columns=['Property_Area'], prefix='Area', drop_first=True)

# Scaling
scale_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Total_Income',
              'EMI', 'Balance_Income', 'Loan_Income_Ratio']
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts(normalize=True))


## 4. Baseline: Simple Train/Test Split

Yeh woh approach hai jo hum normally use karte hain — lekin iska result us specific split par depend karta hai (agar random_state change karein to accuracy bhi change ho sakti hai).


In [ ]:
accuracies = []
for seed in range(5):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    accuracies.append(acc)
    print(f"random_state={seed} -> Accuracy: {acc:.4f}")

print("\nAccuracy varies with each split! Mean:", np.mean(accuracies), "Std:", np.std(accuracies))


## 5. K-Fold Cross-Validation

**Idea:** Data ko K barabar folds mein todo. Har round mein 1 fold test ke liye, baaki (K-1) folds training ke liye. Yeh K baar repeat hota hai, phir results average kiye jate hain.

Isse humein ek zyada **stable aur reliable** performance estimate milta hai, kyunke model K different train/test combinations par test hota hai.


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

model = LogisticRegression(max_iter=1000)
fold_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    score = accuracy_score(y_test, model.predict(X_test))
    fold_scores.append(score)
    print(f"Fold {fold}: Accuracy = {score:.4f}")

print(f"\nMean Accuracy: {np.mean(fold_scores):.4f}")
print(f"Std Deviation: {np.std(fold_scores):.4f}")


In [ ]:
# Same thing, but using sklearn's built-in cross_val_score (shortcut)
scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5, scoring='accuracy')
print("Fold scores:", scores)
print("Mean:", scores.mean(), "Std:", scores.std())


## 6. Stratified K-Fold Cross-Validation

Loan_Status dataset **imbalanced** ho sakta hai (approved vs rejected ka ratio barabar nahi hota). Normal K-Fold mein yeh ho sakta hai ke kisi fold mein ek class bilkul kam ya zyada aa jaye.

**Stratified K-Fold** har fold mein original class-ratio ko maintain karta hai — isliye classification problems ke liye yeh zyada reliable hota hai.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

strat_scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=skf, scoring='accuracy')
print("Stratified K-Fold scores:", strat_scores)
print("Mean:", strat_scores.mean(), "Std:", strat_scores.std())

# Compare class balance across folds
print("\nClass balance per fold:")
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    print(f"Fold {fold} test set -> ", y.iloc[test_idx].value_counts(normalize=True).round(3).to_dict())


## 7. Cross-Validation with Multiple Models

Cross-validation ka ek bohat useful use-case hai: **different models ko fairly compare karna**, sirf ek train/test split ke basis par nahi.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
}

results = {}
for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=skf, scoring='accuracy')
    results[name] = scores
    print(f"{name}: Mean = {scores.mean():.4f}, Std = {scores.std():.4f}")


In [ ]:
plt.figure(figsize=(7, 5))
plt.boxplot(results.values(), tick_labels=results.keys())
plt.title('Model Comparison via Stratified 5-Fold CV')
plt.ylabel('Accuracy')
plt.xticks(rotation=15)
plt.show()


## 8. Leave-One-Out Cross-Validation (Concept + Small Demo)

**Leave-One-Out (LOO)** ek extreme version hai K-Fold ka, jahan **K = number of samples**. Har round mein sirf 1 sample test ke liye, baaki sab training ke liye.

- **Fayda:** Maximum data training ke liye use hota hai
- **Nuqsan:** Bohat slow hai (n rounds chalte hain), isliye sirf chhote datasets ke liye practical hai

Yahan hum demonstration ke liye sirf ek chhota subset (100 rows) use karenge, poore dataset par LOO bohat slow hoga.


In [ ]:
X_small = X.sample(100, random_state=42)
y_small = y.loc[X_small.index]

loo = LeaveOneOut()
loo_scores = cross_val_score(LogisticRegression(max_iter=1000), X_small, y_small, cv=loo, scoring='accuracy')

print(f"Number of LOO rounds: {len(loo_scores)}")
print(f"Mean Accuracy: {loo_scores.mean():.4f}")


## 9. `cross_validate` — Multiple Metrics at Once

`cross_val_score` sirf ek metric deta hai. Agar humein **multiple metrics** (precision, recall, f1, etc.) ek saath chahiye, to `cross_validate` use karte hain.


In [ ]:
scoring = ['accuracy', 'precision', 'recall', 'f1']

cv_results = cross_validate(
    LogisticRegression(max_iter=1000), X, y, cv=skf, scoring=scoring
)

for metric in scoring:
    key = f'test_{metric}'
    print(f"{metric.capitalize():10s} -> Mean: {cv_results[key].mean():.4f}, Std: {cv_results[key].std():.4f}")


## 10. Conclusion

Is notebook mein humne practice ki:

1. Simple train/test split ka result random_state ke sath kaise vary karta hai
2. **K-Fold Cross-Validation** — manually aur `cross_val_score` ke through
3. **Stratified K-Fold** — imbalanced classification ke liye zaroori
4. Cross-validation se **multiple models compare** karna
5. **Leave-One-Out CV** ka concept aur chhota demo
6. `cross_validate` se **multiple metrics** ek saath nikalna

**Key takeaway:** Cross-validation hamesha ek single train/test split se zyada **reliable estimate** deta hai model ki real-world performance ka.

**Next (Day-3):** Hum inhi models ko **Hyperparameter Tuning** (GridSearchCV / RandomizedSearchCV) ke saath cross-validation combine kar ke behtar banayenge.
